# Preprocessing Analysis

Validates that the `ImagePreprocessor` component handles:
- Various aspect ratios (portrait, landscape, square)
- Letterbox padding correctness
- Pixel normalization range
- Dtype and memory layout for ORT compatibility

In [ ]:
import sys

sys.path.insert(0, ".")

import cv2
import matplotlib.pyplot as plt
import numpy as np

from app.components.preprocessor import ImagePreprocessor

pp = ImagePreprocessor(imgsz=640)

In [ ]:
def make_jpeg(h, w, color=(128, 64, 32)) -> bytes:
    img = np.full((h, w, 3), color, dtype=np.uint8)
    _, buf = cv2.imencode(".jpg", img)
    return buf.tobytes()


cases = [
    ("square 640x640", make_jpeg(640, 640)),
    ("landscape 480x640", make_jpeg(480, 640)),
    ("portrait 1080x720", make_jpeg(1080, 720)),
    ("tiny 32x32", make_jpeg(32, 32)),
]

for name, data in cases:
    blob = pp.run(data)
    print(
        f"{name:25s} → shape={blob.shape}  dtype={blob.dtype}  min={blob.min():.3f}  max={blob.max():.3f}  contiguous={blob.flags['C_CONTIGUOUS']}"
    )

## Visualise Letterbox Padding

In [ ]:
from app.components.preprocessor import ImagePreprocessor


def show_letterboxed(h, w, title):
    img = np.zeros((h, w, 3), dtype=np.uint8)
    img[:, :] = [100, 150, 200]  # blue-ish fill to show content
    _, buf = cv2.imencode(".jpg", img)
    blob = ImagePreprocessor(imgsz=640).run(buf.tobytes())
    # Convert NCHW → HWC for display
    vis = np.transpose(blob[0], (1, 2, 0))
    plt.imshow(vis)
    plt.title(title)
    plt.axis("off")


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cases_viz = [(480, 640, "Landscape"), (1080, 720, "Portrait"), (640, 640, "Square")]
for ax, (h, w, title) in zip(axes, cases_viz, strict=False):
    plt.sca(ax)
    show_letterboxed(h, w, title)
plt.tight_layout()
plt.show()